# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [40]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [41]:
import os
import json
import chromadb
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from tavily import TavilyClient
from chromadb.utils import embedding_functions

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.tooling import tool

In [42]:
# Load environment variables
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web

#### Retrieve Game Tool

In [43]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

@tool
def retrieve_game(query: str) -> str:
    results = collection.query(query_texts=[query], n_results=3)
    docs = []
    for meta in results["metadatas"][0]:
        docs.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
        })
    return json.dumps(docs, indent=2)

#### Evaluate Retrieval Tool

In [45]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\n"
        f"Documents:\n{retrieved_docs}"
    )

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format=EvaluationReport,
    )

    report = response.choices[0].message.parsed
    return json.dumps({"useful": report.useful, "description": report.description})

#### Game Web Search Tool

In [46]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> str:
    """Web search: Finds most results on the web using Tavily
    args:
    - question: a question about game industry.
    """
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    response = tavily.search(question, max_results=3)
    results = response.get("results", [])

    formatted = []
    for r in results:
        formatted.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content", "")[:500],
        })
    return json.dumps(formatted, indent=2)

### Agent

In [58]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

SYSTEM_INSTRUCTIONS = """You are UdaPlay, an expert AI research assistant specializing in the video game industry.

Your workflow for every question:
1. Call retrieve_game with the user's question to search the internal database.
2. Call evaluate_retrieval with the question AND the retrieved documents to judge quality.
3. If evaluate_retrieval returns useful=false, call game_web_search to find the answer online.
4. Synthesize all gathered information and provide a clear, structured answer.

Always:
- Cite your source (Internal Database or Web Search)
- Include: Game Name, Platform, Release Year, Publisher, and Genre when available
- Be concise but complete
- If uncertain, say so clearly
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)


In [59]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

def run_query(agent: Agent, query: str, session_id: str = "demo") -> None:
    print(f"QUERY: {query}")

    run = agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state()
    messages = final_state["messages"]

    for msg in messages:
        if hasattr(msg, 'role'):
            if msg.role == "assistant" and hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    args_str = {k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v)
                                for k, v in args.items()}
            elif msg.role == "tool":
                content_preview = str(msg.content)[:100].replace('\n', ' ')


    for msg in reversed(messages):
        if hasattr(msg, 'role') and msg.role == "assistant" and msg.content:
            print(msg.content)
            break
run_query(agent, "When was Pokémon Gold and Silver released?", session_id="session_1")
run_query(agent, "Which one was the first 3D platformer Mario game?", session_id="session_2")
run_query(agent, "Was Mortal Kombat X released for PlayStation 5?", session_id="session_3")

QUERY: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
**Pokémon Gold and Silver** was released in **1999** for the **Game Boy Color**. 

- **Publisher:** Nintendo
- **Genre:** Role-Playing Game (RPG)

This game is notable for being the second generation of Pokémon games, introducing new regions and gameplay mechanics. 

(Source: Internal Database)
QUERY: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool